In [18]:
import pandas as pd
import numpy as np
import random
import folium
import time
import requests
from IPython.display import display, HTML
import json
from google.colab import files

# Catat waktu mulai tepat sebelum proses kalkulasi wilayah dimulai
start_time = time.time()

# 1. Konfigurasi File
files_matriks = {
    'Selatan': 'matriks_jarak_selatan.csv',
    'Timur': 'matriks_jarak_timur.csv',
    'Utara': 'matriks_jarak_utara.csv',
    'Barat': 'matriks_jarak_barat.csv',
    'Pusat': 'matriks_jarak_pusat.csv'
}

# Baca data koordinat
coords_df = pd.read_csv('koordinat_puskesmas (1).csv')

# Tambahkan koordinat dummy untuk UPTD Gudang Farmasi jika belum ada di data koordinat
# (Using the updated coordinates from our previous conversation)
if not coords_df['Nama'].str.contains('Gudang Farmasi', case=False, na=False).any():
    new_row = pd.DataFrame([{'Nama': 'UPTD Gudang Farmasi Surabaya', 'Latitude': -7.3222876, 'Longitude': 112.7717259, 'Klaster': 'Pusat'}])
    coords_df = pd.concat([coords_df, new_row], ignore_index=True)

In [19]:
def get_osrm_route(coordinates):
    """
    Fetches a road-based route from OSRM for a list of coordinates.
    Coordinates should be a list of (latitude, longitude) tuples.
    Returns a list of (latitude, longitude) tuples for the routed path.
    """
    if not coordinates or len(coordinates) < 2:
        return coordinates

    # OSRM expects coordinates in 'longitude,latitude;longitude,latitude' format
    osrm_coords = ';'.join([f"{lon},{lat}" for lat, lon in coordinates])
    osrm_url = f"http://router.project-osrm.org/route/v1/driving/{osrm_coords}?overview=full&geometries=geojson"

    try:
        response = requests.get(osrm_url)
        response.raise_for_status()  # Raise an exception for HTTP errors
        data = response.json()

        if data['code'] == 'Ok' and data['routes']:
            # OSRM returns coordinates as [longitude, latitude], convert to [latitude, longitude]
            routed_path = [[c[1], c[0]] for c in data['routes'][0]['geometry']['coordinates']]
            return routed_path
        else:
            print(f"OSRM error for route: {data.get('code', 'Unknown error')}")
            return coordinates # Fallback to straight lines if routing fails
    except requests.exceptions.RequestException as e:
        print(f"Error connecting to OSRM: {e}")
        return coordinates # Fallback to straight lines if network error occurs
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return coordinates # Fallback for other errors

In [20]:
# 2. Fungsi Ant Colony Optimization (ACO)
def solve_aco(distance_matrix,
              num_ants=30,
              max_iter=150,
              alpha=1.0,
              beta=2.0,
              rho=0.1,
              Q=100):
    n = len(distance_matrix)
    pheromone = np.ones((n, n)) * 0.1
    visibility = np.zeros((n, n))

    # Pre-kalkulasi visibilitas (1/jarak)
    for i in range(n):
        for j in range(n):
            if i != j and distance_matrix[i][j] > 0:
                visibility[i][j] = 1.0 / distance_matrix[i][j]

    best_route = None
    best_dist = float('inf')

    for it in range(max_iter):
        routes = []
        route_lengths = []

        for ant in range(num_ants):
            route = [0] # Selalu mulai dari Node 0 (Gudang Farmasi)
            visited = set([0])

            while len(visited) < n:
                curr = route[-1]
                probs = np.zeros(n)

                # Hitung probabilitas ke node selanjutnya
                for j in range(n):
                    if j not in visited:
                        probs[j] = (pheromone[curr][j] ** alpha) * (visibility[curr][j] ** beta)

                if probs.sum() == 0:
                    unvisited = list(set(range(n)) - visited)
                    next_node = random.choice(unvisited)
                else:
                    probs = probs / probs.sum()
                    next_node = np.random.choice(range(n), p=probs)

                route.append(next_node)
                visited.add(next_node)

            route.append(0) # Kembali ke Gudang Farmasi

            # Kalkulasi jarak total rute semut ini
            dist = 0
            for i in range(len(route)-1):
                dist += distance_matrix[route[i]][route[i+1]]

            routes.append(route)
            route_lengths.append(dist)

            # Simpan rute terbaik global
            if dist < best_dist:
                best_dist = dist
                best_route = route

        # Update Feromon (Evaporasi & Deposit)
        pheromone *= (1 - rho)
        for i in range(num_ants):
            d_tau = Q / route_lengths[i]
            r = routes[i]
            for j in range(len(r)-1):
                pheromone[r[j]][r[j+1]] += d_tau
                pheromone[r[j+1]][r[j]] += d_tau # Asumsi matriks jarak simetris (dua arah)

    return best_route, best_dist

In [21]:
# 3. Eksekusi Algoritma dan Persiapan Peta
results_dist = {}
results_routes = {}
results_runtime = {} # Dictionary untuk menyimpan runtime per wilayah
colors = {'Selatan': 'red', 'Timur': 'blue', 'Utara': 'green', 'Barat': 'purple', 'Pusat': 'orange'}
pusat_peta = [-7.275445, 112.738845]

# Inisialisasi 1 Peta Gabungan
m_gabungan = folium.Map(location=pusat_peta, zoom_start=12, tiles='cartodbpositron')

# Dictionary untuk menyimpan 5 Peta Individual
peta_individual = {}

print("--- SEDANG MENGKALKULASI RUTE TERBAIK ---")
for region, file in files_matriks.items():
    region_start_time = time.time() # Mulai hitung waktu untuk wilayah ini

    df = pd.read_csv(file, index_col=0)
    places = df.index.tolist()
    matrix = df.values

    # Inisialisasi Peta Khusus untuk wilayah ini
    m_region = folium.Map(location=pusat_peta, zoom_start=12, tiles='cartodbpositron')

    # Fix seed agar hasil bisa direproduksi
    np.random.seed(42)
    random.seed(42)

    # Jalankan ACO
    best_route_idx, best_dist = solve_aco(matrix,
                                          num_ants=50,
                                          max_iter=150,
                                          alpha=1.0,
                                          beta=2.0,
                                          rho=0.1,
                                          Q=100)
    route_names = [places[i] for i in best_route_idx]

    region_end_time = time.time() # Hentikan hitungan waktu

    # Simpan hasil komputasi
    results_dist[region] = best_dist
    results_routes[region] = route_names
    results_runtime[region] = region_end_time - region_start_time # Simpan runtime

    # 4. Plotting ke Peta Gabungan & Peta Individual
    route_coords = []
    for place in route_names:
        match = coords_df[coords_df['Nama'].str.strip().str.lower() == place.strip().lower()]
        if not match.empty:
            lat, lon = match.iloc[0]['Latitude'], match.iloc[0]['Longitude']
            route_coords.append((lat, lon))

            # Marker untuk Peta Gabungan
            folium.CircleMarker(
                location=(lat, lon), radius=5, popup=place,
                color=colors[region], fill=True, fill_color=colors[region]
            ).add_to(m_gabungan)

            # Marker untuk Peta Individual
            folium.CircleMarker(
                location=(lat, lon), radius=6, popup=place,
                color=colors[region], fill=True, fill_color=colors[region]
            ).add_to(m_region)

            # Add a distinct marker for the starting point
            if place.strip().lower() == 'uptd gudang farmasi surabaya':
                folium.Marker(
                    location=(lat, lon),
                    icon=folium.Icon(color='darkblue', icon='home', prefix='fa'),
                    popup="Start Point: UPTD Gudang Farmasi Surabaya"
                ).add_to(m_gabungan)
                folium.Marker(
                    location=(lat, lon),
                    icon=folium.Icon(color='darkblue', icon='home', prefix='fa'),
                    popup="Start Point: UPTD Gudang Farmasi Surabaya"
                ).add_to(m_region)

    # Get road-based route using OSRM
    routed_path_osrm = get_osrm_route(route_coords)

    # Tarik Garis Rute (PolyLine)
    if routed_path_osrm:
        # Garis untuk Peta Gabungan
        folium.PolyLine(
            routed_path_osrm, color=colors[region], weight=3, opacity=0.8,
            tooltip=f"Rute {region} ({best_dist:.2f} km)"
        ).add_to(m_gabungan)

        # Garis untuk Peta Individual
        folium.PolyLine(
            routed_path_osrm, color=colors[region], weight=4, opacity=0.9,
            tooltip=f"Rute {region} ({best_dist:.2f} km)"
        ).add_to(m_region)

    # Simpan peta individual ke dalam dictionary
    peta_individual[region] = m_region

--- SEDANG MENGKALKULASI RUTE TERBAIK ---


In [22]:
# 5. Output Hasil Akhir
print("\n--- HASIL JARAK TERPENDEK, RUNTIME, DAN ALUR RUTE ---")
for region, dist in results_dist.items():
    rt = results_runtime[region]
    print(f"\nWilayah {region.upper()} (Jarak: {dist:.2f} km | Runtime: {rt:.2f} detik)")

    # Menggabungkan list nama puskesmas menjadi satu string dengan tanda panah
    alur_rute = " -> ".join(results_routes[region])
    print(f"Rute: {alur_rute}")

# Menampilkan Total Akumulasi
total_akumulasi = sum(results_dist.values())
print(f"\n==================================================")
print(f"TOTAL AKUMULASI SELURUH WILAYAH: {total_akumulasi:.2f} km")
print(f"==================================================\n")

# Menampilkan Runtime Keseluruhan Program
if 'start_time' in locals():
    total_runtime = time.time() - start_time
    print(f"Total Waktu Komputasi Keseluruhan: {total_runtime:.2f} detik\n")


--- HASIL JARAK TERPENDEK, RUNTIME, DAN ALUR RUTE ---

Wilayah SELATAN (Jarak: 35.41 km | Runtime: 5.82 detik)
Rute: UPTD Gudang Farmasi Surabaya -> Puskesmas Sidosermo Surabaya -> Puskesmas Ngagel Rejo Surabaya -> Puskesmas Sawahan Surabaya -> Puskesmas Banyu Urip Surabaya -> Puskesmas Dukuh Kupang Surabaya -> Puskesmas Putat Jaya Surabaya -> Puskesmas Pakis Surabaya -> Puskesmas Wonokromo Surabaya -> Puskesmas Jagir Surabaya -> Puskesmas Jemursari Surabaya -> Puskesmas Siwalankerto Surabaya -> Puskesmas Gayungan Surabaya -> Puskesmas Kebonsari Surabaya -> Puskesmas Kedurus Surabaya -> Puskesmas Balas Klumprik Surabaya -> Puskesmas Wiyung Surabaya -> UPTD Gudang Farmasi Surabaya

Wilayah TIMUR (Jarak: 28.24 km | Runtime: 4.98 detik)
Rute: UPTD Gudang Farmasi Surabaya -> Puskesmas Kalirungkut Surabaya -> Puskesmas Tenggilis Surabaya -> Puskesmas Menur Surabaya -> Puskesmas Pucang Sewu Surabaya -> Puskesmas Mojo Surabaya -> Puskesmas Pacar Keling Surabaya -> Puskesmas Gading Surabaya -

In [23]:
# 6. Menampilkan Peta Langsung di Cell Output
display(HTML("<h3>PETA GABUNGAN (SELURUH WILAYAH)</h3>"))
display(m_gabungan)

for region in files_matriks.keys():
    display(HTML(f"<h3>PETA WILAYAH {region.upper()} (Jarak: {results_dist[region]:.2f} km)</h3>"))
    display(peta_individual[region])

In [24]:
# Prepare data for JSON output
json_output = {}
for region, routes in results_routes.items():
    json_output[region] = routes

# Define the output file name
output_filename = 'rute_per_wilayah.json'

# Save the dictionary to a JSON file
with open(output_filename, 'w') as f:
    json.dump(json_output, f, indent=4)

print(f"File JSON '{output_filename}' telah berhasil dibuat.")

# Provide a download link for the file
files.download(output_filename)

File JSON 'rute_per_wilayah.json' telah berhasil dibuat.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>